In [ ]:
"""
Agent-Based Market Simulation
==============================
Producers : Qsupply, Mcost, FOP, price, willingness_to_sell_at_price
            → profit, SD (supply decision), PD (price decision)
Consumer  : purchasing_power, willingness_to_buy_at_price
            → QD (quantity demanded), PD (price paid)

Runs on Apple Silicon via MPS, falls back to CPU automatically.
"""

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List
from scipy.ndimage import uniform_filter1d

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = (
    torch.device("mps")  if torch.backends.mps.is_available()  else
    torch.device("cuda") if torch.cuda.is_available()           else
    torch.device("cpu")
)
print(f"Device : {DEVICE}")


# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class SimConfig:
    n_producers:     int   = 500
    n_consumers:     int   = 1_000
    n_periods:       int   = 40
    clearing_iters:  int   = 400          # max tâtonnement iterations per period
    clearing_tol:    float = 0.5          # excess demand threshold to stop early
    price_min:       float = 1.0
    price_max:       float = 200.0
    curve_points:    int   = 400          # resolution of S/D curves
    seed:            int   = 42

    # Shock schedule  {period: ("type", magnitude)}
    # "cost"  → multiply all Mcost by magnitude
    # "demand"→ multiply all purchasing_power by magnitude
    # "supply"→ multiply all Qsupply by magnitude
    shocks: Dict[int, tuple] = field(default_factory=lambda: {
        10: ("cost",   1.25),   # 25 % cost increase
        20: ("demand", 1.30),   # 30 % demand boom
        30: ("cost",   0.80),   # 20 % cost reduction
    })


# ─────────────────────────────────────────────────────────────────────────────
#  AGENT INITIALISATION
# ─────────────────────────────────────────────────────────────────────────────
def init_producers(cfg: SimConfig) -> Dict[str, torch.Tensor]:
    """
    Qsupply                  – max units a producer can bring to market
    Mcost                    – marginal cost per unit
    FOP                      – fixed operating / overhead cost
    price                    – producer's posted ask price (updated each period)
    willingness_to_sell_at_price – minimum price ratio above Mcost they require
                                   (e.g. 0.15 means they need at least Mcost * 1.15)
    """
    n = cfg.n_producers
    return {
        "Qsupply":                    torch.rand(n, device=DEVICE) * 80  + 20,    # 20 – 100
        "Mcost":                      torch.rand(n, device=DEVICE) * 35  + 5,     # 5  – 40
        "FOP":                        torch.rand(n, device=DEVICE) * 120 + 20,    # 20 – 140
        "price":                      torch.zeros(n, device=DEVICE),               # set each period
        "willingness_to_sell_at_price": torch.rand(n, device=DEVICE) * 0.40 + 0.05, # 5 – 45 %
    }


def init_consumers(cfg: SimConfig) -> Dict[str, torch.Tensor]:
    """
    purchasing_power           – total budget available
    willingness_to_buy_at_price – fraction of budget they are willing to spend
                                  per unit (reservation price = WTB * budget)
    """
    n = cfg.n_consumers
    return {
        "purchasing_power":           torch.rand(n, device=DEVICE) * 400 + 50,    # 50 – 450
        "willingness_to_buy_at_price": torch.rand(n, device=DEVICE) * 0.55 + 0.15, # 15 – 70 %
    }


# ─────────────────────────────────────────────────────────────────────────────
#  PRODUCER STEP
# ─────────────────────────────────────────────────────────────────────────────
def producer_step(
    producers: Dict[str, torch.Tensor],
    market_price: torch.Tensor,
) -> Dict[str, torch.Tensor]:
    """
    Given a market price scalar, each producer decides:
      PD  – their posted price (Mcost × (1 + willingness_to_sell_at_price))
      SD  – units supplied  (0 if market_price < PD, else scales with margin)
      profit – (market_price − Mcost) × SD − FOP
    """
    # Price decision: the minimum market price each producer is willing to accept
    PD = producers["Mcost"] * (1.0 + producers["willingness_to_sell_at_price"])

    # Update posted price field
    producers["price"] = PD

    # Supply decision: only supply if market price covers their ask
    willing_mask = (market_price >= PD).float()

    # Scale supplied quantity by how attractive the margin is (capped at full capacity)
    margin_ratio = torch.clamp(
        (market_price - producers["Mcost"]) / (producers["Mcost"] + 1e-8),
        min=0.0, max=2.0
    )
    SD = willing_mask * producers["Qsupply"] * margin_ratio

    # Profit
    profit = (market_price - producers["Mcost"]) * SD - producers["FOP"]

    return {"SD": SD, "PD": PD, "profit": profit}


# ─────────────────────────────────────────────────────────────────────────────
#  CONSUMER STEP
# ─────────────────────────────────────────────────────────────────────────────
def consumer_step(
    consumers: Dict[str, torch.Tensor],
    market_price: torch.Tensor,
) -> Dict[str, torch.Tensor]:
    """
    Given a market price scalar, each consumer decides:
      QD – units demanded  (0 if market_price > reservation price)
      PD – price paid      (0 if they don't buy)

    Reservation price = willingness_to_buy_at_price × purchasing_power
    """
    reservation_price = (
        consumers["willingness_to_buy_at_price"] * consumers["purchasing_power"]
    )

    buys = (market_price <= reservation_price).float()

    # Units affordable at this price, capped at 200 per consumer
    QD = buys * torch.clamp(
        torch.floor(consumers["purchasing_power"] / (market_price + 1e-8)),
        max=200.0
    )

    PD = buys * market_price

    return {"QD": QD, "PD": PD}


# ─────────────────────────────────────────────────────────────────────────────
#  MARKET CLEARING  (Walrasian tâtonnement)
# ─────────────────────────────────────────────────────────────────────────────
def clear_market(
    producers: Dict[str, torch.Tensor],
    consumers: Dict[str, torch.Tensor],
    cfg: SimConfig,
    init_log_price: float = 3.5,
) -> tuple:
    """
    Iteratively adjust log(price) until excess demand ≈ 0.
    Uses adaptive step size: shrinks when oscillating around equilibrium.

    Returns
    -------
    eq_price   : equilibrium price scalar tensor
    p_results  : producer result dict at equilibrium
    c_results  : consumer result dict at equilibrium
    n_iters    : iterations taken to converge
    price_path : list of prices tried (for diagnostics)
    """
    log_p = torch.tensor(init_log_price, dtype=torch.float32, device=DEVICE)
    prev_excess = None
    price_path  = []

    for i in range(cfg.clearing_iters):
        price = torch.exp(log_p)
        price_path.append(price.item())

        pr = producer_step(producers, price)
        cr = consumer_step(consumers, price)

        total_S = pr["SD"].sum()
        total_D = cr["QD"].sum()
        excess  = total_D - total_S

        # Adaptive step: smaller when sign flips (we're oscillating)
        if prev_excess is not None and (excess.item() * prev_excess) < 0:
            step = 0.008
        else:
            step = 0.035

        log_p = log_p + step * torch.tanh(excess / (total_S + 1e-8))
        prev_excess = excess.item()

        if abs(excess.item()) < cfg.clearing_tol:
            return torch.exp(log_p), pr, cr, i + 1, price_path

    return torch.exp(log_p), pr, cr, cfg.clearing_iters, price_path


# ─────────────────────────────────────────────────────────────────────────────
#  SUPPLY & DEMAND CURVES  (sweep prices, record aggregate S and D)
# ─────────────────────────────────────────────────────────────────────────────
def build_curves(
    producers: Dict[str, torch.Tensor],
    consumers: Dict[str, torch.Tensor],
    cfg: SimConfig,
) -> tuple:
    prices = torch.linspace(cfg.price_min, cfg.price_max, cfg.curve_points, device=DEVICE)
    supply_vals = torch.zeros(cfg.curve_points, device=DEVICE)
    demand_vals = torch.zeros(cfg.curve_points, device=DEVICE)

    for i, p in enumerate(prices):
        pr = producer_step(producers, p)
        cr = consumer_step(consumers, p)
        supply_vals[i] = pr["SD"].sum()
        demand_vals[i] = cr["QD"].sum()

    return prices.cpu().numpy(), supply_vals.cpu().numpy(), demand_vals.cpu().numpy()


# ─────────────────────────────────────────────────────────────────────────────
#  APPLY SHOCK
# ─────────────────────────────────────────────────────────────────────────────
def apply_shock(
    producers: Dict[str, torch.Tensor],
    consumers: Dict[str, torch.Tensor],
    shock_type: str,
    magnitude: float,
) -> str:
    if shock_type == "cost":
        producers["Mcost"] *= magnitude
        label = f"cost shock ×{magnitude:.2f}"
    elif shock_type == "demand":
        consumers["purchasing_power"] *= magnitude
        label = f"demand shock ×{magnitude:.2f}"
    elif shock_type == "supply":
        producers["Qsupply"] *= magnitude
        label = f"supply shock ×{magnitude:.2f}"
    else:
        label = "unknown shock"
    return label


# ─────────────────────────────────────────────────────────────────────────────
#  MAIN SIMULATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def run_simulation(cfg: SimConfig):
    torch.manual_seed(cfg.seed)
    producers = init_producers(cfg)
    consumers = init_consumers(cfg)

    # History containers
    history = {
        "period":         [],
        "eq_price":       [],
        "total_supply":   [],
        "total_demand":   [],
        "avg_profit":     [],
        "profitable_pct": [],   # % of producers turning a profit
        "buyers_pct":     [],   # % of consumers who bought
        "iters":          [],
        "shock_periods":  [],
        "shock_labels":   [],
    }

    log_p_warm = 3.5   # warm-start the price search each period

    for t in range(1, cfg.n_periods + 1):

        # Apply shock BEFORE clearing
        if t in cfg.shocks:
            shock_type, magnitude = cfg.shocks[t]
            label = apply_shock(producers, consumers, shock_type, magnitude)
            history["shock_periods"].append(t)
            history["shock_labels"].append(label)
            print(f"  ⚡ Period {t:>3}: {label}")

        eq_price, pr, cr, n_iters, _ = clear_market(
            producers, consumers, cfg, init_log_price=log_p_warm
        )
        log_p_warm = torch.log(eq_price).item()

        total_S = pr["SD"].sum().item()
        total_D = cr["QD"].sum().item()

        history["period"].append(t)
        history["eq_price"].append(eq_price.item())
        history["total_supply"].append(total_S)
        history["total_demand"].append(total_D)
        history["avg_profit"].append(pr["profit"].mean().item())
        history["profitable_pct"].append((pr["profit"] > 0).float().mean().item() * 100)
        history["buyers_pct"].append((cr["QD"] > 0).float().mean().item() * 100)
        history["iters"].append(n_iters)

        print(
            f"  Period {t:>3}: P*=${eq_price.item():6.2f}  "
            f"S={total_S:8.0f}  D={total_D:8.0f}  "
            f"avg_profit=${pr['profit'].mean().item():7.2f}  "
            f"iters={n_iters}"
        )

    # Build S/D curves at final period
    p_axis, s_curve, d_curve = build_curves(producers, consumers, cfg)

    return history, p_axis, s_curve, d_curve


# ─────────────────────────────────────────────────────────────────────────────
#  PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
BLUE   = "#1a6faf"
RED    = "#c0392b"
GREEN  = "#1a9a6f"
AMBER  = "#d4860a"
PURPLE = "#7b52ab"
GRAY   = "#6c757d"
LGRAY  = "#e9ecef"


def find_equilibrium_on_curve(prices, supply, demand):
    """Find the index where supply and demand curves are closest."""
    diff = np.abs(supply - demand)
    idx  = np.argmin(diff)
    eq_p = prices[idx]
    eq_q = (supply[idx] + demand[idx]) / 2
    return eq_p, eq_q, idx


def plot_results(history, p_axis, s_curve, d_curve, cfg: SimConfig):
    plt.style.use("seaborn-v0_8-whitegrid")

    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(
        f"Agent-Based Market Simulation  "
        f"({cfg.n_producers} producers · {cfg.n_consumers} consumers · {cfg.n_periods} periods)",
        fontsize=15, fontweight="bold", y=0.99
    )

    gs = gridspec.GridSpec(
        3, 3,
        figure=fig,
        hspace=0.52, wspace=0.35,
        height_ratios=[1.6, 1, 1],
    )

    ax_sd     = fig.add_subplot(gs[0, :2])   # Supply/Demand curves — wide
    ax_price  = fig.add_subplot(gs[0, 2])    # Equilibrium price over time
    ax_qty    = fig.add_subplot(gs[1, :2])   # Supply vs Demand quantities over time
    ax_profit = fig.add_subplot(gs[1, 2])    # Avg profit over time
    ax_buyers = fig.add_subplot(gs[2, :2])   # Buyer & profitable-producer % over time
    ax_iters  = fig.add_subplot(gs[2, 2])    # Convergence speed

    periods        = history["period"]
    shock_periods  = history["shock_periods"]
    shock_labels   = history["shock_labels"]

    def draw_shocks(ax, ymin=None, ymax=None):
        ylim = ax.get_ylim()
        for sp, sl in zip(shock_periods, shock_labels):
            ax.axvline(x=sp, color=AMBER, lw=1.2, ls="--", alpha=0.8, zorder=1)
        # legend entry only once
        if shock_periods:
            ax.axvline(x=shock_periods[0], color=AMBER, lw=1.2, ls="--",
                       alpha=0.8, label="shock", zorder=1)

    # ── 1. Supply & Demand curves ─────────────────────────────────────────────
    s_smooth = uniform_filter1d(s_curve, size=6)
    d_smooth = uniform_filter1d(d_curve, size=6)

    ax_sd.plot(p_axis, s_smooth, color=BLUE, lw=2.5, label="Supply")
    ax_sd.plot(p_axis, d_smooth, color=RED,  lw=2.5, label="Demand")

    eq_p, eq_q, _ = find_equilibrium_on_curve(p_axis, s_smooth, d_smooth)
    ax_sd.axvline(x=eq_p, color=GREEN, lw=1.4, ls=":", alpha=0.85)
    ax_sd.axhline(y=eq_q, color=GREEN, lw=1.4, ls=":", alpha=0.85)
    ax_sd.scatter([eq_p], [eq_q], s=130, color=GREEN, zorder=6,
                  label=f"P* = ${eq_p:.1f},  Q* = {eq_q:.0f}")
    ax_sd.annotate(
        f"P* = ${eq_p:.1f}\nQ* = {eq_q:.0f}",
        xy=(eq_p, eq_q),
        xytext=(eq_p + (p_axis[-1] - p_axis[0]) * 0.06, eq_q + max(s_smooth) * 0.06),
        fontsize=9.5, color=GREEN,
        arrowprops=dict(arrowstyle="->", color=GREEN, lw=1.1),
    )
    ax_sd.set_xlabel("Price ($)", fontsize=10)
    ax_sd.set_ylabel("Aggregate Quantity", fontsize=10)
    ax_sd.set_title("Supply & Demand curves (final period)", fontsize=11, fontweight="bold")
    ax_sd.legend(fontsize=9.5)
    ax_sd.set_xlim(p_axis[0], p_axis[-1])
    ax_sd.set_ylim(bottom=0)

    # ── 2. Equilibrium price over time ────────────────────────────────────────
    ax_price.plot(periods, history["eq_price"], color=BLUE, lw=2, marker="o", markersize=3.5)
    ax_price.fill_between(periods, history["eq_price"], alpha=0.12, color=BLUE)
    for sp in shock_periods:
        ax_price.axvline(x=sp, color=AMBER, lw=1.2, ls="--", alpha=0.8)
    ax_price.set_xlabel("Period", fontsize=10)
    ax_price.set_ylabel("Price ($)", fontsize=10)
    ax_price.set_title("Equilibrium price", fontsize=11, fontweight="bold")

    # ── 3. Total supply vs total demand over time ─────────────────────────────
    ax_qty.plot(periods, history["total_supply"], color=BLUE, lw=2,
                marker="o", markersize=3, label="Total supply (SD)")
    ax_qty.plot(periods, history["total_demand"], color=RED,  lw=2,
                marker="o", markersize=3, label="Total demand (QD)")
    for sp in shock_periods:
        ax_qty.axvline(x=sp, color=AMBER, lw=1.2, ls="--", alpha=0.8)
    ax_qty.set_xlabel("Period", fontsize=10)
    ax_qty.set_ylabel("Aggregate units", fontsize=10)
    ax_qty.set_title("Supply vs Demand over time", fontsize=11, fontweight="bold")
    ax_qty.legend(fontsize=9)

    # ── 4. Average producer profit over time ──────────────────────────────────
    profits = history["avg_profit"]
    bar_colors = [GREEN if p >= 0 else RED for p in profits]
    ax_profit.bar(periods, profits, color=bar_colors, alpha=0.82, width=0.7)
    ax_profit.axhline(y=0, color=GRAY, lw=1)
    for sp in shock_periods:
        ax_profit.axvline(x=sp, color=AMBER, lw=1.2, ls="--", alpha=0.8)
    ax_profit.set_xlabel("Period", fontsize=10)
    ax_profit.set_ylabel("Avg profit ($)", fontsize=10)
    ax_profit.set_title("Avg producer profit", fontsize=11, fontweight="bold")

    # ── 5. Participation rates over time ──────────────────────────────────────
    ax_buyers.plot(periods, history["buyers_pct"],     color=RED,    lw=2,
                   marker="s", markersize=3, label="Consumers buying (%)")
    ax_buyers.plot(periods, history["profitable_pct"], color=GREEN,  lw=2,
                   marker="^", markersize=3.5, label="Profitable producers (%)")
    for sp in shock_periods:
        ax_buyers.axvline(x=sp, color=AMBER, lw=1.2, ls="--", alpha=0.8)
    ax_buyers.set_ylim(0, 105)
    ax_buyers.set_xlabel("Period", fontsize=10)
    ax_buyers.set_ylabel("% of agents", fontsize=10)
    ax_buyers.set_title("Market participation rates", fontsize=11, fontweight="bold")
    ax_buyers.legend(fontsize=9)

    # ── 6. Convergence (iterations to clear market) ───────────────────────────
    ax_iters.bar(periods, history["iters"], color=PURPLE, alpha=0.75, width=0.7)
    for sp in shock_periods:
        ax_iters.axvline(x=sp, color=AMBER, lw=1.2, ls="--", alpha=0.8)
    ax_iters.set_xlabel("Period", fontsize=10)
    ax_iters.set_ylabel("Iterations", fontsize=10)
    ax_iters.set_title("Clearing convergence speed", fontsize=11, fontweight="bold")

    # ── Shared shock legend ───────────────────────────────────────────────────
    if shock_periods:
        shock_text = "  |  ".join(
            f"t={sp}: {sl}" for sp, sl in zip(shock_periods, shock_labels)
        )
        fig.text(0.5, -0.005, f"⚡ Shocks — {shock_text}",
                 ha="center", fontsize=9, color=AMBER)

    plt.savefig("market_simulation.png", dpi=150, bbox_inches="tight")
    print("\nPlot saved → market_simulation.png")
    plt.show()


# ─────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    cfg = SimConfig(
        n_producers    = 500,
        n_consumers    = 1_000,
        n_periods      = 40,
        clearing_iters = 400,
        clearing_tol   = 0.5,
        seed           = 42,
        shocks         = {
            10: ("cost",   1.25),   # production costs up 25%
            20: ("demand", 1.30),   # consumer budgets up 30%
            30: ("cost",   0.80),   # production costs down 20%
        },
    )

    print("=" * 65)
    print(" Agent-Based Market Simulation")
    print(f" Producers : {cfg.n_producers}")
    print(f" Consumers : {cfg.n_consumers}")
    print(f" Periods   : {cfg.n_periods}")
    print(f" Device    : {DEVICE}")
    print("=" * 65)

    history, p_axis, s_curve, d_curve = run_simulation(cfg)
    plot_results(history, p_axis, s_curve, d_curve, cfg)